In [ ]:
!pip install transformers torch pandas requests nltk rouge-score --quiet

In [ ]:
import requests
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM
from rouge_score import rouge_scorer

In [ ]:
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [ ]:
API_KEY = "YOUR_API_KEY"  # optional

url = f"https://newsapi.org/v2/top-headlines?country=in&pageSize=5&apiKey={API_KEY}"

articles = []

try:
    response = requests.get(url)
    data = response.json()

    if 'articles' in data:
        articles = [a.get('content') for a in data['articles'] if a.get('content')]
    else:
        print("⚠️ API issue, using fallback data...")
except:
    print("⚠️ API failed, using fallback data...")

# fallback (guaranteed)
if not articles:
    articles = [
        "India is seeing rapid growth in AI technology across multiple sectors including healthcare and education.",
        "The stock market showed significant improvement today with tech companies leading the gains.",
        "Scientists have discovered a new method to improve battery efficiency using advanced materials.",
        "Climate change continues to impact global weather patterns and environmental stability.",
        "New advancements in space technology are enabling deeper exploration of the universe."
    ]

df = pd.DataFrame(articles, columns=['text'])
df.head()

⚠️ API issue, using fallback data...


,text
0,India is seeing rapid growth in AI technology ...
1,The stock market showed significant improvemen...
2,Scientists have discovered a new method to imp...
3,Climate change continues to impact global weat...
4,New advancements in space technology are enabl...


In [ ]:
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = re.sub(r'http\S+', '', text)      # remove links
    text = re.sub(r'[^a-zA-Z ]', '', text)   # remove special characters
    words = text.lower().split()             # lowercase + split
    words = [w for w in words if w not in stop_words]  # remove stopwords
    return " ".join(words)

df['clean_text'] = df['text'].apply(clean_text)

df.head()

,text,clean_text
0,India is seeing rapid growth in AI technology ...,india seeing rapid growth ai technology across...
1,The stock market showed significant improvemen...,stock market showed significant improvement to...
2,Scientists have discovered a new method to imp...,scientists discovered new method improve batte...
3,Climate change continues to impact global weat...,climate change continues impact global weather...
4,New advancements in space technology are enabl...,new advancements space technology enabling dee...


In [ ]:
print("⏳ Loading models (robust version, no pipeline)...")

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# -------------------------------
# BART MODEL
# -------------------------------
bart_name = "facebook/bart-large-cnn"

bart_tokenizer = AutoTokenizer.from_pretrained(bart_name)
bart_model = AutoModelForSeq2SeqLM.from_pretrained(bart_name)

print("✅ BART loaded")

# -------------------------------
# T5 MODEL
# -------------------------------
t5_name = "t5-small"

t5_tokenizer = AutoTokenizer.from_pretrained(t5_name)
t5_model = AutoModelForSeq2SeqLM.from_pretrained(t5_name)

print("✅ T5 loaded")

print("🎉 All models loaded successfully!")

⏳ Loading models (robust version, no pipeline)...


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

✅ BART loaded


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

✅ T5 loaded
🎉 All models loaded successfully!


In [ ]:
# -------------------------------
# BART Summarization Function
# -------------------------------
def summarize_bart(text):
    inputs = bart_tokenizer.encode(
        text[:1000],
        return_tensors="pt",
        max_length=1024,
        truncation=True
    )

    summary_ids = bart_model.generate(
        inputs,
        max_length=80,
        min_length=20,
        length_penalty=2.0,
        num_beams=4
    )

    return bart_tokenizer.decode(summary_ids[0], skip_special_tokens=True)


# -------------------------------
# T5 Summarization Function
# -------------------------------
def summarize_t5(text):
    text = "summarize: " + text

    inputs = t5_tokenizer.encode(
        text[:1000],
        return_tensors="pt",
        max_length=512,
        truncation=True
    )

    summary_ids = t5_model.generate(
        inputs,
        max_length=80,
        min_length=20,
        num_beams=4
    )

    return t5_tokenizer.decode(summary_ids[0], skip_special_tokens=True)

In [ ]:
df['bart_summary'] = df['clean_text'].apply(summarize_bart)
df['t5_summary'] = df['clean_text'].apply(summarize_t5)

df[['text', 'bart_summary', 't5_summary']]

,text,bart_summary,t5_summary
0,India is seeing rapid growth in AI technology ...,india seeing rapid growth ai technology across...,india seeing rapid growth ai technology across...
1,The stock market showed significant improvemen...,stock market showed significant improvement to...,stock market showed significant improvement to...
2,Scientists have discovered a new method to imp...,scientists discovered new method improve batte...,scientists discovered new method improve batte...
3,Climate change continues to impact global weat...,climate change continues to impact global weat...,climate change continues impact global weather...
4,New advancements in space technology are enabl...,new advancements space technology enabling dee...,new advancements space technology enabling dee...


In [ ]:
scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)

def rouge_calc(original, summary):
    score = scorer.score(original, summary)
    return score['rouge1'].fmeasure

df['bart_score'] = df.apply(lambda x: rouge_calc(x['text'], x['bart_summary']), axis=1)
df['t5_score'] = df.apply(lambda x: rouge_calc(x['text'], x['t5_summary']), axis=1)

df[['bart_score', 't5_score']]

,bart_score,t5_score
0,0.685714,0.615385
1,0.647059,0.606061
2,0.685714,0.645161
3,0.740741,0.666667
4,0.666667,0.615385


In [ ]:
df['best_model'] = df.apply(lambda x: "BART" if x['bart_score'] > x['t5_score'] else "T5", axis=1)

df[['text', 'best_model']]

,text,best_model
0,India is seeing rapid growth in AI technology ...,BART
1,The stock market showed significant improvemen...,BART
2,Scientists have discovered a new method to imp...,BART
3,Climate change continues to impact global weat...,BART
4,New advancements in space technology are enabl...,BART


In [ ]:
import os
os.listdir()

['.config', 'sample_data']